# Контроль: воспроизводится ли нулевая точка

Образ Kaggle обновляется без предупреждения. За один день 4 октября
`transformers` сменился с 5.0.0 на 5.16.1.

Таблица рецепта сравнивает этапы попарно, и весь её смысл в том, что между
двумя строками отличается **только рецепт**. Если между ними отличается
ещё и версия библиотеки, часть разницы окажется её.

Поэтому перед тем как дописывать в журнал, посчитанный на другом образе,
считается заново то, что в нём уже есть, и сравнивается парным тестом.

**Сошлось** — журнал можно продолжать, и это пишется в отчёт.
**Разошлось** — прогон отказывается работать и говорит, что рецепт надо
считать заново целиком.

В журнал не пишется ничего. Это проверка, а не этап.

**Три с половиной минуты карты.** Считается одна оценка корпуса,
62 594 фрагмента.

In [ ]:
import os
for ключ in ('HF_HUB_OFFLINE', 'TRANSFORMERS_OFFLINE', 'HF_DATASETS_OFFLINE'):
    os.environ[ключ] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

import time
НАЧАЛО = time.time()

In [ ]:
!nvidia-smi --query-gpu=name --format=csv,noheader

import torch, transformers, sentence_transformers
print('transformers', transformers.__version__)
print('sentence-transformers', sentence_transformers.__version__)
print('эти версии и есть то, что проверяется')
assert torch.cuda.is_available(), 'нет карты'

In [ ]:
import glob, gzip, shutil, zipfile


def find_source(root='/kaggle/input', max_depth=5):
    found, archives = [], []
    for cur, dirs, files in os.walk(root):
        if cur.count('/') - root.count('/') >= max_depth:
            dirs.clear()
        if 'rufin' in dirs:
            found.append((1 if '__output__.json' in files else 0, cur))
        archives += [os.path.join(cur, f) for f in files if f.endswith('.zip')]
    if found:
        found.sort()
        return found[0][1], None
    return (os.path.dirname(archives[0]), archives[0]) if archives else (None, None)


src, archive = find_source()
assert src, 'датасет с кодом не подключён'
if archive:
    with zipfile.ZipFile(archive) as z:
        z.extractall('/kaggle/working')
else:
    for имя in os.listdir(src):
        путь, цель = os.path.join(src, имя), os.path.join('/kaggle/working', имя)
        if os.path.isdir(путь):
            shutil.copytree(путь, цель, dirs_exist_ok=True)
        elif имя.endswith(('.py', '.md')):
            shutil.copyfile(путь, цель)

os.makedirs('/kaggle/working/chunks', exist_ok=True)
dst = '/kaggle/working/chunks/base.jsonl'
if os.path.exists(dst):
    os.remove(dst)
найдено = (glob.glob('/kaggle/input/**/chunks_base.jsonl', recursive=True)
           + glob.glob('/kaggle/input/**/chunks_base.jsonl.gz', recursive=True))
assert найдено, 'нет chunks_base.jsonl'
opener = gzip.open if найдено[0].endswith('.gz') else open
with opener(найдено[0], 'rb') as fi, open(dst, 'wb') as fo:
    shutil.copyfileobj(fi, fo, length=1 << 20)
n = sum(1 for _ in open(dst, encoding='utf-8'))
assert n == 62594, f'ожидалось 62594, а не {n}'

журнал = glob.glob('/kaggle/input/**/journal.json', recursive=True)
assert журнал, 'нет journal.json: без него проверять нечего'
os.makedirs('/kaggle/working/train', exist_ok=True)
shutil.copyfile(журнал[0], '/kaggle/working/train/journal.json')
import json
было = json.load(open(журнал[0], encoding='utf-8'))
print('журнал:', [э['tag'] for э in было['этапы']])

In [ ]:
import subprocess

cmd = ['python', 'run_phase_d.py', '--step', 'контроль',
       '--batch-size', '128',
       '--journal', '/kaggle/working/train/journal.json',
       '--runs', '/kaggle/working/kontrol']
print('запуск:', ' '.join(cmd), flush=True)
код = subprocess.run(cmd, cwd='/kaggle/working').returncode
print('\nкод возврата:', код, '| прошло', f'{(time.time()-НАЧАЛО)/60:.1f} мин')

# Код 1 здесь означает «разошлось», и это осмысленный ответ, а не поломка.
# Поэтому тетрадь не валится, а печатает вывод.
import json, os
путь = '/kaggle/working/kontrol/kontrol.json'
if os.path.exists(путь):
    print(json.dumps(json.load(open(путь, encoding='utf-8')),
                     ensure_ascii=False, indent=1))

shutil.rmtree('/kaggle/working/chunks', ignore_errors=True)